[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L09.ipynb)

# Lecture 09 — Autograd & Backpropagation by Hand

*The library does the calculus — and then you do it by hand once, to see that it is not magic.*

**PyTorch from Ground Up · Volume I · Chapters 14–15** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Turn on gradient tracking and describe the computation graph the forward pass builds.
- Explain what `.backward()` does: the chain rule, node by node, right to left.
- Know where gradients live (leaf tensors), and that they accumulate.
- Switch tracking off with `torch.no_grad()` and `.detach()`.
- Backpropagate through a tiny network by hand and confirm it against autograd.
- Explain the dying-ReLU problem from a single local derivative.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Autograd in one sentence

`grad = 2 * w` works for one parameter. For a million, you need **autograd**. Last lecture computed the slope by hand. Automatic differentiation computes it for every parameter at once.

<sub>Source: lecture slides 3–7</sub>

**`tape.py`** — predict the output, then run.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
print(x.requires_grad)          # True
y = x ** 2                  # node: power
z = 2 * y + 1               # nodes: multiply, add

<details><summary>▸ Expected output</summary>

```
True
```
</details>

## 02 · .backward() and the chain rule

`.backward()` walks the graph **right to left** and leaves the slope in `x.grad`. z = 2x² + 1, so dz/dx = 4x — and at x = 3 that’s 12. PyTorch got it right without a single derivative formula.

<sub>Source: lecture slides 8–11</sub>

**`backward.py`** — predict the output, then run.

In [ ]:
z.backward()
print(x.grad)               # tensor(12.)

<details><summary>▸ Expected output</summary>

```
tensor(12.)
```
</details>

🧪 *Interactive lab in the app:* **Forward builds the graph; backward walks it** — try it there to see this idea move.

## 03 · Where gradients live

Only **leaf tensors** keep their `.grad`. The tensors you created with `requires_grad=True` store their gradient; intermediates like y and z don’t, to save memory. Need one? Call `.retain_grad()` on it *before* `.backward()`.

<sub>Source: lecture slides 12–13</sub>

**`retain.py`** — predict the output, then run.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
y.retain_grad()                # keep y's gradient
z = y * 5
z.backward()
print(x.grad)           # 60.0       (dz/dx = 15x² = 60)
print(y.grad)           # 5.0        (dz/dy = 5)

<details><summary>▸ Expected output</summary>

```
tensor(60.)
tensor(5.)
```
</details>

## 04 · Why gradients accumulate

`.grad` is **added to, not replaced**. Call `.backward()` twice and the second result piles on the first.

<sub>Source: lecture slides 14–16</sub>

🧪 *Interactive lab in the app:* **Three backward calls, one tensor** — try it there to see this idea move.

**`accumulate.py`** — predict the output, then run.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f"step {i}    grad = {x.grad}")
    # x.grad.zero_()    ← uncomment to fix

<details><summary>▸ Expected output</summary>

```
step 0    grad = 6.0
step 1    grad = 12.0
step 2    grad = 18.0
```
</details>

> ⚠️ **Watch out — Forgetting to zero gradients**  
> Forgetting to zero gradients is the single most common autograd bug. If your training loss stops improving or behaves erratically, check that `zero_grad()` is called before every `backward()`.
>
> The fix is `x.grad.zero_()` on a bare tensor, or `optimizer.zero_grad()` inside a training loop.

## 05 · Switching tracking off

Evaluating, not learning? **Turn the tape off.** Inside `torch.no_grad()` no graph is built — no time or memory wasted on gradients you’ll never use.

<sub>Source: lecture slides 17–19</sub>

**`no_grad_detach.py`** — predict the output, then run.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
    y = x ** 2
    print(y.requires_grad)          # False

y = x ** 2
y_val = y.detach()
print(y_val)                # tensor(9.)
print(y_val.requires_grad)  # False

<details><summary>▸ Expected output</summary>

```
False
tensor(9.)
False
```
</details>

## 06 · Many parameters, one call

**One call** filled the gradient for both w and b — and would for a million. Note x has no `requires_grad`: it is data, not a parameter.

<sub>Source: lecture slides 21–22</sub>

**`many_params.py`** — predict the output, then run.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)
y = w * x + b               # y = 2·3 + 1 = 7
loss = (y - 5) ** 2         # (7-5)² = 4
loss.backward()
print(w.grad)       # 12.0      (dloss/dw)
print(b.grad)       # 4.0      (dloss/db)

<details><summary>▸ Expected output</summary>

```
tensor(12.)
tensor(4.)
```
</details>

## 07 · Backprop by hand: the tiny network

Now do it once **by hand**, without the safety net. The smallest possible network, one data point, the loss — then walk backward with pencil and paper. After this, backprop stops being a black box.

<sub>Source: lecture slides 24–34</sub>

**`hand_calc.py`** — predict the output, then run.

In [ ]:
x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)
y = torch.tensor(2.0)
# Forward
h = w * x + b
a = torch.relu(h)
loss = (a - y) ** 2
# Backward
loss.backward()
print(f"loss = {loss.item()}")
print(f"dL/dw = {w.grad.item()}")
print(f"dL/db = {b.grad.item()}")

<details><summary>▸ Expected output</summary>

```
loss = 9.0
dL/dw = 12.0
dL/db = 6.0
```
</details>

🧪 *Interactive lab in the app:* **The tiny network — every gradient, live** — try it there to see this idea move.

## 08 · When the ReLU gate closes

Now set x = −1. Then h = −4 and the ReLU outputs 0. **Predict the gradients** before you run it.

<sub>Source: lecture slides 35–40</sub>

**`dead_relu.py`** — predict the output, then run.

In [ ]:
x = torch.tensor(-1.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)
y = torch.tensor(2.0)
h = w * x + b            # h = -4
a = torch.relu(h)        # a = 0
loss = (a - y) ** 2      # (0 - 2)² = 4
loss.backward()
print(f"h = {h.item()}, a = {a.item()}")
print(f"dL/dw = {w.grad.item()}")
print(f"dL/db = {b.grad.item()}")

<details><summary>▸ Expected output</summary>

```
h = -4.0, a = 0.0
dL/dw = -0.0
dL/db = 0.0
```
</details>

> ⚠️ **Watch out — The dead neuron problem**  
> When a ReLU neuron’s input is negative for every data point, its gradient is permanently zero and it stops learning. This is the “dead neuron” problem.
>
> Alternatives like `LeakyReLU` allow a small gradient even for negative inputs — covered in the layer catalog, Lecture 12.

## The rule — worth remembering

**Autograd — the six rules**

1. Set `requires_grad=True` on tensors whose gradient you need (model parameters).
2. The forward pass builds a computation graph. `backward()` walks it in reverse to compute gradients.
3. Gradients are stored in `.grad` on leaf tensors only.
4. Gradients accumulate. Always call `zero_grad()` before `backward()`.
5. Use `torch.no_grad()` during evaluation to skip tracking and save memory.
6. Use `.detach()` to get a tensor’s value without its gradient history.

**Backpropagation — the five rules**

1. Backpropagation is just the chain rule applied one node at a time, right to left.
2. At each node: incoming gradient × local derivative = outgoing gradient.
3. ReLU’s local derivative is 1 (if input > 0) or 0 (if input ≤ 0). Zero blocks the gradient entirely.
4. Autograd does exactly this calculation, automatically, for every parameter at once.
5. Doing it by hand once is worth a hundred readings of the theory.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Ch 14 · Ex 1

Compute the gradient of y = 5x³ + 3x² − 2x + 7 at x = 2 using autograd. Verify by hand.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

dy/dx = 15x² + 6x − 2 = 60 + 12 − 2 = **70**. `x = torch.tensor(2.0, requires_grad=True); y = 5*x**3 + 3*x**2 - 2*x + 7; y.backward(); x.grad` → `tensor(70.)`.
</details>

### Exercise 02 · Ch 14 · Ex 4

Wrap a computation in `torch.no_grad()`, confirm `requires_grad=False`, then call `.backward()` on the result. What error do you get?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn`. No graph was recorded inside the block, so there is nothing to walk backward through.
</details>

### Exercise 03 · Ch 15 · Ex 1

Repeat the hand calculation with x = 1, w = −2, b = 3, y = 0. Compute h, a, L, then ∂L/∂w and ∂L/∂b by hand. Verify with autograd.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

h = −2·1 + 3 = 1; a = ReLU(1) = 1; L = (1 − 0)² = 1. ∂L/∂a = 2; ∂a/∂h = 1 (h > 0); ∂L/∂h = 2. **∂L/∂w = 2 × x = 2**, **∂L/∂b = 2**. Load the preset in the tiny-network lab to check.
</details>

### Exercise 04 · Ch 15 · Ex 4

In the dead-ReLU example, change the bias to b = 5.0 so h becomes positive. Do the gradients become nonzero? Confirm with code.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Yes. h = 3·(−1) + 5 = 2 > 0, so the gate opens: a = 2, L = 0 — and here the gradients are 2(a − y) = 0 because the prediction happens to be exactly right. Use y = 1 to see non-zero values: ∂L/∂a = 2, ∂L/∂w = 2 × (−1) = −2, ∂L/∂b = 2.
</details>

### Interview-style

Explain what `loss.backward()` does under the hood, and why `zero_grad()` must be called before it in a training loop. Three to four sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

During the forward pass PyTorch records every operation on tensors that require gradients, building a computation graph. `loss.backward()` walks that graph in reverse from the loss, and at each node multiplies the incoming gradient by the node’s local derivative — the chain rule — until it reaches the leaf parameters, where the result is stored in `.grad`. Those stored gradients are added to, not overwritten, so without `zero_grad()` each step would mix in the gradients of every previous step. The updates would then be wrong and growing, and training would diverge.
</details>